# Statistical Analysis
Testing business hypotheses using statistical methods.

In [ ]:
import pandas as pd
import sqlite3
from scipy import stats

In [ ]:
conn = sqlite3.connect('../ecommerce.db')
query = """
SELECT 
    c.customer_id,
    COUNT(DISTINCT o.order_id) as order_count,
    SUM(oi.revenue) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id
"""
customer_data = pd.read_sql(query, conn)
conn.close()
customer_data.head()

## Hypothesis: Do repeat customers spend significantly more overall than one-time customers?
- **H0:** There is no significant difference in total revenue between one-time and repeat customers.
- **H1:** Repeat customers spend significantly more.

In [ ]:
one_time = customer_data[customer_data['order_count'] == 1]['total_revenue']
repeat = customer_data[customer_data['order_count'] > 1]['total_revenue']

t_stat, p_val = stats.ttest_ind(repeat, one_time, equal_var=False)
print(f'T-statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4e}')

if p_val < 0.05:
    print('Reject H0: There is a significant difference in revenue.')
else:
    print('Fail to reject H0: No significant difference.')